INSTALAÇÃO DE FERRAMENTAS

In [1]:
%%capture
!pip install -U eemont git+https://github.com/andrebelem/geemap-tools.git # fazendo a instalação em silêncio !

In [2]:
import ee
import geemap
import geemap_tools as gee
import geopandas as gpd
import pandas as pd


In [3]:
ee.Authenticate()
ee.Initialize(project='ee-silasjunior') #ALTERAR PARA SEU PROJETO (todos tem acesso editor à esse Notebook)

LER O KML

In [12]:
# Aqui, coloque seu arquivo kml do computador
kml = gpd.read_file('/content/Bracuí.kml')

In [13]:
# Garantir WGS84

if kml.crs and kml.crs.to_epsg() != 4326:
    kml = kml.to_crs(4326)
kml

,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,geometry
0,None,Bracuí,None,NaT,NaT,NaT,None,1,0,-1,NaN,None,"POLYGON Z ((-44.37944 -22.957 0, -44.37914 -22..."


In [14]:
# Forçar 2D (remover Z) — funciona em Shapely 1.8 e 2.x
import shapely.wkb

def force_2d(geom):
    if getattr(geom, "has_z", False):
        # método WKB é o mais confiável para qualquer tipo (Polygon/MultiPolygon)
        return shapely.wkb.loads(shapely.wkb.dumps(geom, output_dimension=2))
    return geom

In [16]:
# Aplicar a função force_2d para remover a coordenada Z da coluna 'geometry'
kml['geometry'] = kml['geometry'].apply(force_2d)

# Verificar se a coordenada Z foi removida
# Isso pode ser feito inspecionando a geometria de uma linha
print("Geometria após remover Z:")
print(kml.loc[0, 'geometry'])

Geometria após remover Z:
POLYGON ((-44.37944358139906 -22.9569977487161, -44.37913755474098 -22.95516754214118, -44.37902094484875 -22.95397746200019, -44.37895264763437 -22.95228805060437, -44.37904036916598 -22.95003532786995, -44.37905964610331 -22.94913628417759, -44.37908220250391 -22.94860976368972, -44.37932090090584 -22.94800615010091, -44.37939930942924 -22.94762786140561, -44.37940372882646 -22.9459421105674, -44.37946465454493 -22.944562207436, -44.37945842094688 -22.94411639972285, -44.38101739963669 -22.94414125937639, -44.38165980862743 -22.94381866482539, -44.38227123346054 -22.94264744934393, -44.38241440349552 -22.94180527457419, -44.38208464038147 -22.94116697241221, -44.38175369677617 -22.94052638247372, -44.38152664473075 -22.93973481021247, -44.38170177108929 -22.9388899359179, -44.38262255888284 -22.93787133330612, -44.38309027523944 -22.93718676557717, -44.38299779869814 -22.9364995550171, -44.38269092603662 -22.93583437216351, -44.38266271775566 -22.93576031018

In [18]:
# Criar um mapa interativo usando geemap
Map = geemap.Map(center=[-22.95, -44.39], zoom=12) # Ajuste o centro e o zoom conforme a sua área de interesse

# Adicionar o GeoDataFrame ao mapa
Map.add_gdf(kml, layer_name="Bracuí KML")

# Exibir o mapa
Map

Map(center=[-22.95, -44.39], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

CALCULO NDVI

In [20]:
kml_ee = geemap.geopandas_to_ee(kml)
# Se você precisar apenas da geometria da FeatureCollection, use:
kml_ee_geometry = kml_ee.geometry()
print(kml_ee.getInfo())
print(kml_ee_geometry.getInfo())

{'type': 'FeatureCollection', 'columns': {'Name': 'String', 'altitudeMode': 'Object', 'begin': 'Object', 'description': 'Object', 'drawOrder': 'Object', 'end': 'Object', 'extrude': 'Integer', 'icon': 'Object', 'id': 'Object', 'system:index': 'String', 'tessellate': 'Integer', 'timestamp': 'Object', 'visibility': 'Integer'}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-44.37944358139906, -22.9569977487161], [-44.37913755474098, -22.95516754214118], [-44.37902094484875, -22.95397746200019], [-44.37895264763437, -22.95228805060437], [-44.37904036916598, -22.95003532786995], [-44.37905964610331, -22.94913628417759], [-44.37908220250391, -22.94860976368972], [-44.37932090090584, -22.94800615010091], [-44.37939930942924, -22.94762786140561], [-44.37940372882646, -22.9459421105674], [-44.37946465454493, -22.944562207436], [-44.37945842094688, -22.94411639972285], [-44.38101739963669, -22.94414125937639], [-44.38165980862743, -22.94381866482539], [-44.382

In [21]:
# Coleção Sentinel-2 de Nível 2A (Correção Atmosférica)
s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(kml_ee)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) # Filtro de nuvens
          .sort('system:time_start', False)) # Ordena da mais recente para a mais antiga

recent_image = s2_col.first().clip(kml_ee)

In [22]:
ndvi = recent_image.normalizedDifference(['B8', 'B4']).rename('NDVI')

In [50]:
Map = geemap.Map()
Map.centerObject(kml_ee, 14)

ndvi_vis = {
    'min': 0,
    'max': 1,
    'palette': ['red', 'yellow', 'green']
}

Map.addLayer(recent_image, {'bands': ['B4', 'B3', 'B2'], 'max': 3000}, 'Imagem Sentinel-2 (RGB)')
Map.addLayer(ndvi, ndvi_vis, 'NDVI Recente')
Map

Map(center=[-22.944534477353937, -44.388953095537886], controls=(WidgetControl(options=['position', 'transpare…